# Multimodal Messages with LangChain

This reference notebook demonstrates how to send text, image, and audio content to LangChain agents using structured `HumanMessage` content blocks.

## Learning goals

- Configure an agent and send a text-only message.
- Upload an image, convert it to base64, and include it in a multimodal message.
- Record audio, encode it as base64 WAV data, and send it to an audio-capable model.
- Understand the relationship between a content block's `type`, encoded data, and MIME type.

## Before you run the notebook

1. Set the required model-provider API key in your environment or `.env` file.
2. Run the cells from top to bottom so variables such as `agent`, `img_b64`, and `aud_b64` exist before they are used.
3. The image section requires an uploaded image, and the audio section requires a working microphone and audio input device.

Each request follows the same pattern: create a `HumanMessage`, place it in the agent state under `messages`, invoke the agent, and print the final message content.

In [21]:
# Load API keys and other local settings from the project's .env file.
from dotenv import load_dotenv

load_dotenv()

# LangSmith tracing is opt-in. Set LANGSMITH_TRACING=true and provide
# LANGSMITH_API_KEY in .env when you want runs to appear in LangSmith.
import os

tracing_enabled = os.getenv("LANGSMITH_TRACING", "false").lower() == "true"
if tracing_enabled:
    if not os.getenv("LANGSMITH_API_KEY"):
        raise ValueError(
            "LANGSMITH_TRACING=true, but LANGSMITH_API_KEY is not configured."
        )
    os.environ.setdefault(
        "LANGSMITH_ENDPOINT", "https://api.smith.langchain.com"
    )
else:
    # Prevent inherited LangSmith settings from causing offline upload warnings.
    os.environ["LANGSMITH_TRACING"] = "false"
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    os.environ.pop("LANGSMITH_API_KEY", None)
    os.environ.pop("LANGCHAIN_API_KEY", None)

# Restart the notebook kernel after changing .env so old callbacks unload.

In [22]:
from langchain.agents import create_agent

# The system prompt establishes the agent's role and response style.
# gpt-5-nano is used for the text and image examples.
agent = create_agent(
    model="gpt-5-nano",
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [23]:
from langchain.messages import HumanMessage

# A message content list allows text and other modalities to be sent together.
question = HumanMessage(
    content=[
        {"type": "text", "text": "What is the capital of The Moon?"}
    ]
)

# Agent state expects a list of messages under the "messages" key.
response = agent.invoke({"messages": [question]})

# The final message is the agent's answer after it processes the request.
print(response["messages"][-1].content)

In this fictional setting, the Moon has a capital named Selene Prime.

Selene Prime
- Location: Nestled along the rim of Shackleton Crater in the Moon’s south polar region, where a ring of solar reflectors (the Solar Spine) keeps a steady ballet of daylight across the city. Permanent shade pockets exist in the crater floor, but the rim enjoys near-constant solar access.
- Government: The Lunar Federation’s seat of power. The Grand Rotunda hosts sessions of the Lunar Council, with districts sending delegates. A Prime Secretary serves as the executive, supported by ministries for Resources, Science, Commerce, and Culture.
- Population: A bustling metropolis of several million residents, with a steady influx of scientists, engineers, miners, diplomats, and artists from lunar settlements and visiting Earth allies.
- Economy: The city is a hub for governance and high-end research, lunar ice mining operations run through the hinterlands, helium-3 trade, solar-energy export, and cutting-edge 

## Image Input

The image workflow has three steps: collect an image with a notebook widget, encode its bytes as base64 text, and attach that data to a `HumanMessage` alongside an instruction.

In [24]:
from ipywidgets import FileUpload
from IPython.display import display

# Restrict the widget to one image so the following cells have one clear input.
uploader = FileUpload(accept="image/*", multiple=False)
display(uploader)

FileUpload(value=(), accept='image/*', description='Upload')

In [31]:
# Inspect the widget value to confirm that an image has been selected.
print(uploader.value)

({'name': 'images.jpeg', 'type': 'image/jpeg', 'size': 49129, 'content': <memory at 0x11bf9f100>, 'last_modified': datetime.datetime(2026, 9, 19, 13, 14, 49, 821000, tzinfo=datetime.timezone.utc)},)


In [32]:
import base64

if not uploader.value:
    print(
        "No image uploaded. Select an image in the widget above and re-run this cell "
        "before running the next image cell."
    )
else:
    # FileUpload stores the selected file as a one-item tuple of dictionaries.
    uploaded_file = uploader.value[0]

    # The content field is a memoryview, so convert it to bytes before encoding it.
    img_bytes = bytes(uploaded_file["content"])
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")

    # Preserve the uploaded MIME type so JPEG, PNG, and other image formats work.
    image_mime_type = uploaded_file["type"]
    print(f"Prepared {uploaded_file['name']} ({image_mime_type}) for the next cell.")

Prepared images.jpeg (image/jpeg) for the next cell.


In [33]:
if not uploader.value:
    print("Image analysis skipped because no image has been uploaded yet.")
else:
    # Image input requires a vision-capable model. The text agent above uses
    # gpt-5-nano, which does not accept image_url content in this setup.
    image_agent = create_agent(
        model="gpt-4o",
        system_prompt="You are a science fiction writer describing images for the user.",
    )

    # Put the natural-language instruction and image data in the same message.
    multimodal_question = HumanMessage(
        content=[
            {"type": "text", "text": "Tell me about this capital"},
            {
                "type": "image",
                "base64": img_b64,
                "mime_type": image_mime_type,
            },
        ]
    )

    response = image_agent.invoke({"messages": [multimodal_question]})
    print(response["messages"][-1].content)

This imagined lunar capital is a sprawling hub of human innovation and futuristic architecture nestled on the Moon’s surface. Transparent domes house habitats and laboratories, allowing for stunning views of the starry void and the distant Earth. These structures are interconnected by a network of tunnels designed to protect inhabitants from the harsh, airless lunar environment.

Solar panels stretch across sections of the landscape, harnessing the Sun's energy to power the colony. Nearby, sleek landing pads accommodate spacecraft, highlighting the capital’s role as a central point for exploration and transportation. This Moon capital is a testament to human ingenuity and the spirit of exploration, a beacon of civilization on a distant world.


### Image section summary

The image bytes remain local until they are placed in the message. Base64 makes the binary data transportable as text, while `mime_type` tells the model how to interpret those bytes. The text block supplies the task; the image block supplies the evidence.

The image example uses `gpt-4o`, a vision-capable model. The text agent uses `gpt-5-nano`, which is kept separate because it does not accept image content in this configuration.

## Audio Input

The audio workflow records a short WAV clip, stores it in memory, base64-encodes it, and sends it to an audio-capable model. This cell requires microphone permissions and a functioning input device.

In [34]:
import base64
import io
import time

import sounddevice as sd
from scipy.io.wavfile import write
from tqdm import tqdm

In [35]:
# Recording settings: mono audio at CD-quality sample rate.
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(
    int(duration * sample_rate),
    samplerate=sample_rate,
    channels=1,
)

# sd.rec starts asynchronously, so wait for the recording to finish.
# The progress bar is only visual feedback during the five-second capture.
for _ in tqdm(range(duration * 10), desc="Recording"):
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write the NumPy audio array to an in-memory WAV file.
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

# Models receive the binary WAV data as base64 text in the message block.
aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


Recording: 100%|██████████| 50/50 [00:05<00:00,  9.57it/s]


Done.


In [36]:
# Use an audio-capable model for the audio content block.
agent = create_agent(model="gpt-audio")

multimodal_question = HumanMessage(
    content=[
        {"type": "text", "text": "Tell me about this audio file"},
        {
            "type": "audio",
            "base64": aud_b64,
            "mime_type": "audio/wav",
        },
    ]
)

response = agent.invoke({"messages": [multimodal_question]})
print(response["messages"][-1].content)

I can help analyze the content of the audio by providing details such as language, main topics, or overall tone. Let me start by listening to it. I'll analyze it now.


### Audio section summary

The recording is converted to a WAV byte stream before base64 encoding. The `audio` content block and `audio/wav` MIME type distinguish this payload from the image example, while the surrounding message structure stays the same.

## Conclusion and reusable pattern

This notebook uses one consistent pattern across modalities:

1. Prepare the input and identify its MIME type.
2. Encode binary data as base64 text when required by the content block.
3. Build a `HumanMessage` with a text instruction and one or more modality blocks.
4. Invoke the agent with `{"messages": [message]}`.
5. Read the final assistant response from `response["messages"][-1].content`.

For future experiments, keep the message structure stable and change only the model, content block, or instruction. Text requests use `gpt-5-nano`, image requests use the vision-capable `gpt-4o`, and audio requests use `gpt-audio`. Model availability, supported modalities, API credentials, microphone permissions, and payload limits can vary by provider, so verify those requirements before adapting this notebook.